# Hannah · 02 — SVD Baseline

Collaborative-filtering baseline for the **Bayesian Personalized Meal Recommendation System**.
Personal working notebook (Hannah) — the shared `02_svd_baseline.ipynb` stays untouched.

**Plan**

1. Load the provided splits; exclude 0-star rows (comment-without-stars, see EDA §9)
2. Naive baselines — global mean, user mean, shrunken item mean
3. Carve a **warm-item holdout** out of train (EDA §11: the provided val/test are 100% item-cold-start, so SVD can only be judged on warm items)
4. Extended hyperparameter grid on a warm-train subsample, then retrain the winner on full warm train — plus a **BaselineOnly (bias-only) ablation** trained beside it
5. Evaluate RMSE/MAE in both regimes, labeled; add a **temporal (last-interaction) warm holdout** to measure how much the random split flatters the model
6. Ranking: leave-one-out HR@10 / NDCG@10 (positives = 5-star holds) for SVD vs bias-only vs popularity, with 95% CIs — warm-only protocol
7. Save models + predictions + chosen params + ranking metrics to `outputs/` for the hybrid notebook


## 0 · Setup

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
assert (ROOT / "config.yaml").exists(), "config.yaml not found - run from the repo or notebooks/"
CFG = yaml.safe_load((ROOT / "config.yaml").read_text())
DATA = ROOT / CFG["paths"]["data_dir"]
OUT = ROOT / CFG["paths"]["outputs_dir"]
MODELS = ROOT / CFG["paths"]["models_dir"]
SEED = CFG["data"]["random_seed"]

# The zero-rating policy is a modeling decision, not decoration — fail fast if it drifts.
assert CFG["data"]["zero_rating_policy"] == "exclude", (
    "hannah_02 assumes data.zero_rating_policy == 'exclude' (0 = comment without stars)")

# Shared metric helpers — single implementation across the hannah_* notebooks.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.evaluation.hannah_metrics import (
    hr_at_k, mae, mean_ci_normal, ndcg_at_k, rank_of_positive, rmse, wilson_ci)

# Same chart conventions as notebook 01
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 11.5,
    "axes.labelcolor": INK2, "legend.frameon": False, "figure.dpi": 100,
})


## 1 · Load splits, drop 0-star rows

Zeros mean "left a comment without awarding stars", not "hated it" — excluded from rating
training per the EDA. The splits ship contiguous internal ids `u` / `i` for matrix work.

In [2]:
train = pd.read_csv(DATA / "interactions_train.csv", parse_dates=["date"])
validation = pd.read_csv(DATA / "interactions_validation.csv", parse_dates=["date"])
test = pd.read_csv(DATA / "interactions_test.csv", parse_dates=["date"])

starred = train[train.rating > 0].copy()
print(f"train rows: {len(train):,}  |  starred (rating>0): {len(starred):,} "
      f"({100 * len(starred) / len(train):.1f}%)")
print(f"users: {starred.user_id.nunique():,}  recipes: {starred.recipe_id.nunique():,}")

# Quoted throughout: the five-star share of the set the models actually train on.
FIVE_STAR_SHARE = 100 * float((starred.rating == 5).mean())
print(f"five-star share of the modeled (starred) training set: {FIVE_STAR_SHARE:.1f}%")


train rows: 698,901  |  starred (rating>0): 681,944 (97.6%)
users: 24,961  recipes: 159,131
five-star share of the modeled (starred) training set: 76.0%


## 2 · Naive baselines

The starred training set is heavily five-star-skewed (exact share printed in §1), so SVD
must beat these to mean anything. Item means get shrinkage (EDA §17: half the catalog has
a "perfect" average resting on one vote).


In [3]:
K_SHRINK = 20  # pseudo-observations pulling item means toward the global mean

def fit_baselines(frame):
    g = frame.rating.mean()
    u = frame.groupby("user_id").rating.mean()
    stats = frame.groupby("recipe_id").rating.agg(["sum", "count"])
    i = (stats["sum"] + K_SHRINK * g) / (stats["count"] + K_SHRINK)
    return g, u, i

GLOBAL_MEAN, user_mean, item_mean_shrunk = fit_baselines(starred)

def predict_baseline(df, kind):
    if kind == "global":
        return np.full(len(df), GLOBAL_MEAN)
    if kind == "user_mean":
        return df.user_id.map(user_mean).fillna(GLOBAL_MEAN).to_numpy()
    if kind == "item_mean":
        return df.recipe_id.map(item_mean_shrunk).fillna(GLOBAL_MEAN).to_numpy()

def rmse_mae(y_true, y_pred):
    """(RMSE, MAE) via the shared src.evaluation.hannah_metrics implementations."""
    return rmse(y_true, y_pred), mae(y_true, y_pred)

print(f"global mean: {GLOBAL_MEAN:.4f} | users with a mean: {user_mean.size:,} "
      f"| shrunken item means: {item_mean_shrunk.size:,}")
# All three are scored on the warm holdout and the provided splits in §5.
# The stats are refit on warm_train in §3 so the baselines never see held-out rows.


global mean: 4.6878 | users with a mean: 24,961 | shrunken item means: 159,131


## 3 · Warm-item holdout carved from train

The provided val/test recipes never appear in train, so a vanilla SVD's item factors are
undefined there — evaluating on those files mostly measures bias terms. For a fair SVD score:
hold out one starred rating per multi-rating user, keeping only holdout rows whose recipe
still appears in the remaining train (warm items).

In [4]:
multi = starred.groupby("user_id").user_id.transform("size") >= 2
candidates = starred[multi]
held_idx = candidates.groupby("user_id").sample(n=1, random_state=SEED).index

warm_train = starred.drop(index=held_idx)
held = starred.loc[held_idx]
warm_holdout = held[held.recipe_id.isin(set(warm_train.recipe_id))]

print(f"warm_train: {len(warm_train):,} rows")
print(f"holdout: {len(held):,} held rows -> {len(warm_holdout):,} warm "
      f"({100 * len(warm_holdout) / len(held):.1f}% of held rows have a warm recipe)")

# Refit baseline statistics on warm_train only — with one held rating per user, leaving
# it inside user_mean would leak exactly the row we score on.
GLOBAL_MEAN, user_mean, item_mean_shrunk = fit_baselines(warm_train)


warm_train: 657,500 rows
holdout: 24,444 held rows -> 23,603 warm (96.6% of held rows have a warm recipe)


## 4 · Tune + train Surprise SVD (+ bias-only ablation)

Extended grid — the base 3×3 `n_factors ∈ {50, 100, 150} × reg_all ∈ {0.02, 0.05, 0.1}`
plus the lower-capacity / heavier-shrinkage configs `(25, 0.1), (25, 0.2), (50, 0.2)` —
at fixed `n_epochs=20, lr_all=0.005`, scored by RMSE on an **inner** per-user holdout
carved (same recipe as §3) from a seeded 200K-row subsample of `warm_train`, to bound
runtime. The base grid's winner sat at the corner `(50, 0.1)`; the extension checks
whether that was a true optimum or an artifact of the search boundary — the printout says
whether the final winner is interior. The winner is retrained on the **full** `warm_train`
and used everywhere downstream, alongside a Surprise `BaselineOnly` (ALS defaults)
bias-only model — the "is latent structure real?" ablation.


In [5]:
import time
from itertools import product

from surprise import SVD, BaselineOnly, Dataset, Reader

reader = Reader(rating_scale=(CFG["data"]["rating_min"], CFG["data"]["rating_max"]))

# Seeded 200K-row subsample of warm_train; carve the inner per-user warm holdout from it
tune_pool = warm_train.sample(n=200_000, random_state=SEED)
inner_multi = tune_pool.groupby("user_id").user_id.transform("size") >= 2
inner_idx = tune_pool[inner_multi].groupby("user_id").sample(n=1, random_state=SEED).index
inner_train = tune_pool.drop(index=inner_idx)
inner_hold = tune_pool.loc[inner_idx]
inner_hold = inner_hold[inner_hold.recipe_id.isin(set(inner_train.recipe_id))]
print(f"grid fit rows: {len(inner_train):,} | inner warm holdout: {len(inner_hold):,}")

inner_trainset = Dataset.load_from_df(
    inner_train[["user_id", "recipe_id", "rating"]], reader).build_full_trainset()

# Base 3x3 grid plus an extension past its corner winner (50, 0.1): fewer factors and
# heavier shrinkage, to test whether that winner was a search-boundary artifact.
grid_configs = list(product([50, 100, 150], [0.02, 0.05, 0.1]))
grid_configs += [(25, 0.1), (25, 0.2), (50, 0.2)]

grid_rows = []
for n_factors, reg_all in grid_configs:
    t0 = time.time()
    cand = SVD(n_factors=n_factors, n_epochs=20, lr_all=0.005, reg_all=reg_all,
               random_state=SEED)
    cand.fit(inner_trainset)
    preds = np.array([cand.predict(u, r).est
                      for u, r in zip(inner_hold.user_id, inner_hold.recipe_id)])
    rmse_val, _ = rmse_mae(inner_hold.rating, preds)
    grid_rows.append((n_factors, reg_all, rmse_val, time.time() - t0))

grid = pd.DataFrame(grid_rows, columns=["n_factors", "reg_all", "inner RMSE", "seconds"])
BEST = grid.loc[grid["inner RMSE"].idxmin()]
BEST_PARAMS = {"n_factors": int(BEST.n_factors), "n_epochs": 20, "lr_all": 0.005,
               "reg_all": float(BEST.reg_all)}

# Interior = the winner does not sit on the boundary of the tried values on either axis.
tried_nf = sorted({nf for nf, _ in grid_configs})
tried_reg = sorted({rg for _, rg in grid_configs})
WINNER_INTERIOR = bool(tried_nf[0] < BEST_PARAMS["n_factors"] < tried_nf[-1]
                       and tried_reg[0] < BEST_PARAMS["reg_all"] < tried_reg[-1])
print(f"winner: n_factors={BEST_PARAMS['n_factors']}, reg_all={BEST_PARAMS['reg_all']} "
      f"| interior to the searched ranges: {WINNER_INTERIOR}")
print(f"config.yaml default: n_factors={CFG['svd']['n_factors']}, "
      f"reg_all={CFG['svd']['reg_all']} (outputs/hannah_svd_params.json is authoritative)")
grid.pivot(index="n_factors", columns="reg_all", values="inner RMSE").round(4)


grid fit rows: 187,609 | inner warm holdout: 10,278


winner: n_factors=25, reg_all=0.2 | interior to the searched ranges: False
config.yaml default: n_factors=50, reg_all=0.1 (outputs/hannah_svd_params.json is authoritative)


reg_all,0.02,0.05,0.10,0.20
n_factors,,,,
25,NaN,NaN,0.6798,0.6793
50,0.6828,0.6822,0.6815,0.6807
100,0.6847,0.6838,0.6829,NaN
150,0.6879,0.6867,0.6854,NaN


In [6]:
svd_trainset = Dataset.load_from_df(
    warm_train[["user_id", "recipe_id", "rating"]], reader).build_full_trainset()

algo = SVD(random_state=SEED, **BEST_PARAMS)
algo.fit(svd_trainset)

# Bias-only ablation: global mean + regularized user/item biases, ALS defaults.
# If SVD cannot beat this, the latent factors are not adding real structure.
bias_algo = BaselineOnly(bsl_options={"method": "als"})
bias_algo.fit(svd_trainset)

print(f"final SVD {BEST_PARAMS} retrained on full warm_train: "
      f"{svd_trainset.n_ratings:,} ratings")
print("BaselineOnly (ALS defaults) trained on the same warm_train")


Estimating biases using als...


final SVD {'n_factors': 25, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.2} retrained on full warm_train: 657,500 ratings
BaselineOnly (ALS defaults) trained on the same warm_train


## 5 · Evaluate — warm vs cold regimes, labeled

- **Warm** (`warm_holdout`): the honest SVD number.
- **Bias-only ablation** (`BaselineOnly`): global mean + regularized user/item biases.
  The SVD-vs-bias gap is the measure of whether latent structure is real here.
- **Cold** (provided validation/test): with item factors undefined, Surprise collapses to
  global mean + regularized user bias — a *shrunken* user mean. Report it anyway; it is
  the regime the Bayesian layer targets in notebook 04.

§5a stress-tests the warm number with a temporal split; ranking follows in §5b.
RMSE rewards calibrated scores, but a recommender must also *rank*.


In [7]:
def surprise_predict(model, df):
    return np.array([model.predict(u, r).est for u, r in zip(df.user_id, df.recipe_id)])

def svd_predict(df):
    return surprise_predict(algo, df)

rows = []
for regime, frame in [("warm holdout", warm_holdout),
                      ("cold (provided validation)", validation[validation.rating > 0]),
                      ("cold (provided test)", test[test.rating > 0])]:
    preds = {"global": predict_baseline(frame, "global"),
             "user mean": predict_baseline(frame, "user_mean"),
             "item mean (shrunk)": predict_baseline(frame, "item_mean"),
             "bias only (ALS)": surprise_predict(bias_algo, frame),
             "SVD": svd_predict(frame)}
    for model, p in preds.items():
        r_val, m_val = rmse_mae(frame.rating, p)
        rows.append((regime, model, r_val, m_val))

results = pd.DataFrame(rows, columns=["regime", "model", "RMSE", "MAE"])
results.style.format({"RMSE": "{:.4f}", "MAE": "{:.4f}"}).hide(axis="index")


regime,model,RMSE,MAE
warm holdout,global,0.7273,0.4988
warm holdout,user mean,0.7899,0.4420
warm holdout,item mean (shrunk),0.7175,0.4762
warm holdout,bias only (ALS),0.6958,0.4434
warm holdout,SVD,0.6936,0.4393
cold (provided validation),global,0.9127,0.6280
cold (provided validation),user mean,0.8876,0.5493
cold (provided validation),item mean (shrunk),0.9127,0.6280
cold (provided validation),bias only (ALS),0.8722,0.5890
cold (provided validation),SVD,0.8658,0.5793


### 5a · Temporal warm holdout — does the random split flatter the model?

The §3 holdout stars a *random* rating per multi-rating user, so training routinely
contains interactions that happened *after* the row being predicted — information a
deployed system would not have. For a project explicitly claiming to model *evolving*
preferences, that is the wrong direction of information flow. Two checks:

1. **Measure the leak**: what share of random-holdout targets have strictly-later
   same-user interactions inside `warm_train`? (Printed below — recomputed, not quoted.)
2. **Rebuild the split temporally**: hold each multi-rating user's chronologically
   **last** starred rating (by date, ties broken by row order), retrain the same-winner
   SVD and the bias-only model on the temporal train, and compare RMSE/MAE side by side.

Expectation: the temporal split is harder, i.e. the random split overstates warm accuracy.


In [8]:
# (1) Leakage stat on the RANDOM split: share of warm_holdout targets whose user has at
# least one strictly-later interaction inside warm_train (vectorized via per-user max date).
user_last_train_date = warm_train.groupby("user_id").date.max()
PCT_LATER = 100 * float(
    (warm_holdout.user_id.map(user_last_train_date) > warm_holdout.date).mean())
print(f"random-split leakage: {PCT_LATER:.1f}% of warm_holdout targets have strictly-later "
      f"same-user train interactions")

# (2) Temporal split: hold each multi-rating user's chronologically LAST starred rating.
# Stable sort by (user_id, date) keeps original row order within equal dates, so tail(1)
# breaks date ties by row order.
temporal_held_idx = (starred[multi]
                     .sort_values(["user_id", "date"], kind="stable")
                     .groupby("user_id").tail(1).index)
temporal_train = starred.drop(index=temporal_held_idx)
temporal_held = starred.loc[temporal_held_idx]
temporal_holdout = temporal_held[temporal_held.recipe_id.isin(set(temporal_train.recipe_id))]
print(f"temporal_train: {len(temporal_train):,} rows | temporal warm holdout: "
      f"{len(temporal_holdout):,} rows "
      f"({100 * len(temporal_holdout) / len(temporal_held):.1f}% of held rows warm)")

# Same-winner SVD + BaselineOnly retrained on the temporal train
temporal_trainset = Dataset.load_from_df(
    temporal_train[["user_id", "recipe_id", "rating"]], reader).build_full_trainset()
algo_temporal = SVD(random_state=SEED, **BEST_PARAMS)
algo_temporal.fit(temporal_trainset)
bias_temporal = BaselineOnly(bsl_options={"method": "als"})
bias_temporal.fit(temporal_trainset)

temporal_rows = []
for model_name, rand_model, temp_model in [("SVD", algo, algo_temporal),
                                           ("bias only (ALS)", bias_algo, bias_temporal)]:
    rr, rm = rmse_mae(warm_holdout.rating, surprise_predict(rand_model, warm_holdout))
    tr, tm = rmse_mae(temporal_holdout.rating, surprise_predict(temp_model, temporal_holdout))
    temporal_rows.append((model_name, rr, rm, tr, tm, tr - rr))

temporal_results = pd.DataFrame(
    temporal_rows, columns=["model", "random RMSE", "random MAE",
                            "temporal RMSE", "temporal MAE", "delta RMSE (temporal-random)"])
temporal_results.style.format(
    {c: "{:.4f}" for c in temporal_results.columns[1:]}).hide(axis="index")


random-split leakage: 75.1% of warm_holdout targets have strictly-later same-user train interactions


temporal_train: 657,500 rows | temporal warm holdout: 23,597 rows (96.5% of held rows warm)


Estimating biases using als...


model,random RMSE,random MAE,temporal RMSE,temporal MAE,delta RMSE (temporal-random)
SVD,0.6936,0.4393,0.6923,0.4306,-0.0014
bias only (ALS),0.6958,0.4434,0.6945,0.4340,-0.0013


### 5b · Ranking — leave-one-out HR@10 / NDCG@10 (warm only)

For a seeded sample of `warm_holdout` users whose held rating is **5 stars** (relevance =
"loved", not merely "rated"), rank the held-out recipe among 99 seeded negatives the user
has never interacted with — the `seen` exclusion uses **all** train interactions,
including 0-star comment rows. Models: SVD, bias-only, raw train popularity. HR@10 gets a
Wilson 95% CI; NDCG@10 a normal-approximation 95% CI (helpers from
`src.evaluation.hannah_metrics`). This is a **warm-only** protocol — every candidate comes
from the `warm_train` pool; cold-regime ranking is notebook 04's job.

**Reading:** an RMSE-trained SVD predicts ≈5 almost everywhere, so it barely ranks, while
held-out positives skew popular and negatives are drawn uniformly — expect popularity to
win. Exact numbers with CIs are printed in the table and in the §7 summary.


In [9]:
rng_rank = np.random.default_rng(SEED)
N_NEG, K = 99, 10
N_USERS_TARGET = 3_000

pool = warm_train.recipe_id.unique()
seen = train.groupby("user_id").recipe_id.agg(set)  # ALL train interactions, incl. rating==0
pop = warm_train.recipe_id.value_counts()

# Positives = loved (5-star) held rows; one held row per user, so rows == users.
loved_hold = warm_holdout[warm_holdout.rating == 5]
sample_users = loved_hold.sample(n=min(N_USERS_TARGET, len(loved_hold)), random_state=SEED)
N_USERS = len(sample_users)
print(f"5-star holdout rows: {len(loved_hold):,} of {len(warm_holdout):,} "
      f"| sampled users: {N_USERS:,}")

def sample_negatives(rated, positive):
    negs = np.empty(0, dtype=pool.dtype)
    while negs.size < N_NEG:
        draw = pool[rng_rank.integers(0, pool.size, size=3 * N_NEG)]
        draw = pd.unique(np.concatenate([negs, draw]))
        keep = np.fromiter(((r not in rated) and (r != positive) for r in draw),
                           bool, len(draw))
        negs = draw[keep]
    return negs[:N_NEG]

MODEL_ORDER = ["svd", "bias_only", "popularity"]
rank_records = {m: [] for m in MODEL_ORDER}
for row in sample_users.itertuples():
    negs = sample_negatives(seen.get(row.user_id, set()), row.recipe_id)
    cands = np.concatenate(([row.recipe_id], negs))
    scores = {
        "svd": np.array([algo.predict(row.user_id, r).est for r in cands]),
        "bias_only": np.array([bias_algo.predict(row.user_id, r).est for r in cands]),
        "popularity": pop.reindex(cands).fillna(0).to_numpy(dtype=float),
    }
    for m in MODEL_ORDER:
        rank_records[m].append(rank_of_positive(scores[m][0], scores[m][1:], rng=rng_rank))

PROTOCOL = "warm, positives=rating5, 1+99"
metric_rows = []
for m in MODEL_ORDER:
    ranks_m = np.asarray(rank_records[m])
    hits = hr_at_k(ranks_m, K)
    ndcgs = ndcg_at_k(ranks_m, K)
    hr_lo, hr_hi = wilson_ci(hits.sum(), N_USERS)
    nd_lo, nd_hi = mean_ci_normal(ndcgs)
    metric_rows.append((m, float(hits.mean()), float(ndcgs.mean()),
                        hr_lo, hr_hi, nd_lo, nd_hi, N_USERS, PROTOCOL))

ranking_metrics = pd.DataFrame(metric_rows, columns=[
    "model", "hr_at_10", "ndcg_at_10", "hr_lo95", "hr_hi95",
    "ndcg_lo95", "ndcg_hi95", "n_users", "protocol"])
ranking_metrics.style.format(
    {c: "{:.4f}" for c in ranking_metrics.columns[1:7]}).hide(axis="index")


5-star holdout rows: 18,473 of 23,603 | sampled users: 3,000


model,hr_at_10,ndcg_at_10,hr_lo95,hr_hi95,ndcg_lo95,ndcg_hi95,n_users,protocol
svd,0.3350,0.1844,0.3183,0.3521,0.1738,0.1951,3000,"warm, positives=rating5, 1+99"
bias_only,0.4147,0.2576,0.3972,0.4324,0.2449,0.2703,3000,"warm, positives=rating5, 1+99"
popularity,0.6977,0.5012,0.6810,0.7138,0.4867,0.5156,3000,"warm, positives=rating5, 1+99"


## 6 · Save artifacts for the hybrid notebook

Models (SVD + bias-only), prediction CSVs for every split — random and temporal warm
holdouts, cold validation/test — each with `svd_pred`, `bias_pred` and an `in_eval` flag
(`rating > 0`), the chosen params + provenance JSON (including the leakage stat and the
temporal-model settings), and the regenerated warm ranking metrics.


In [10]:
import json
import pickle

MODELS.mkdir(parents=True, exist_ok=True)
with open(MODELS / "hannah_svd.pkl", "wb") as f:
    pickle.dump(algo, f)
with open(MODELS / "hannah_baseline_only.pkl", "wb") as f:
    pickle.dump(bias_algo, f)
saved = [MODELS / "hannah_svd.pkl", MODELS / "hannah_baseline_only.pkl"]

def write_preds(frame, path, svd_model, bias_model):
    out = frame[["user_id", "recipe_id", "rating"]].copy()
    out["svd_pred"] = surprise_predict(svd_model, frame)
    out["bias_pred"] = surprise_predict(bias_model, frame)
    out["in_eval"] = frame.rating.to_numpy() > 0
    out.to_csv(path, index=False)
    return path

for name, frame in [("warm_holdout", warm_holdout), ("validation", validation), ("test", test)]:
    saved.append(write_preds(frame, OUT / f"hannah_svd_preds_{name}.csv", algo, bias_algo))
saved.append(write_preds(temporal_holdout, OUT / "hannah_svd_preds_warm_holdout_temporal.csv",
                         algo_temporal, bias_temporal))

params_out = {
    **BEST_PARAMS, "random_state": SEED,
    "selected_by": ("inner per-user holdout RMSE on a seeded 200K-row warm_train subsample; "
                    "extended grid = {50,100,150}x{0.02,0.05,0.1} plus "
                    "(25,0.1), (25,0.2), (50,0.2)"),
    "inner_rmse": round(float(BEST["inner RMSE"]), 4),
    "winner_interior": WINNER_INTERIOR,
    "pct_holdout_with_later_train_interaction": round(PCT_LATER, 3),
    "temporal_model": {
        "svd_params": {**BEST_PARAMS, "random_state": SEED},
        "baseline_only": "surprise BaselineOnly, ALS defaults",
        "split": ("per multi-rating user: chronologically last starred rating held "
                  "(by date, ties by row order); holdout restricted to warm recipes"),
        "train_rows": int(len(temporal_train)),
        "holdout_rows": int(len(temporal_holdout)),
    },
}
(OUT / "hannah_svd_params.json").write_text(json.dumps(params_out, indent=2))
saved.append(OUT / "hannah_svd_params.json")

ranking_metrics.to_csv(OUT / "hannah_ranking_metrics.csv", index=False)
saved.append(OUT / "hannah_ranking_metrics.csv")
print("saved:", ", ".join(p.name for p in saved))


saved: hannah_svd.pkl, hannah_baseline_only.pkl, hannah_svd_preds_warm_holdout.csv, hannah_svd_preds_validation.csv, hannah_svd_preds_test.csv, hannah_svd_preds_warm_holdout_temporal.csv, hannah_svd_params.json, hannah_ranking_metrics.csv


## 7 · Takeaways

All numbers live in the printed summary below — recomputed on every run, so nothing here
can go stale. Qualitative reading:

- **Hyperparameters** — the extended grid tests whether the base grid's corner winner
  `(50, 0.1)` was a boundary artifact; the printout states the winner and whether it is
  interior to the searched ranges. With signal spread thin across ~159K recipes, low
  capacity + heavy shrinkage is the expected regime.
- **Warm vs bias-only** — the SVD-vs-BaselineOnly gap measures whether latent structure
  adds anything beyond user/item biases; the printout quantifies it in RMSE *and* MAE.
- **Temporal honesty** — the random per-user holdout lets training see later same-user
  activity (leakage share printed); the temporal last-interaction split removes that and
  is the more deployment-faithful warm number for a system claiming to model evolving
  preferences.
- **Cold regime** — the provided val/test are 100% item-cold: SVD degenerates to a
  shrunken user mean there, and the printout says explicitly which model wins cold MAE.
  Closing the warm-cold gap with item-side priors is the job of notebooks 03/04.
- **Ranking** — warm-only protocol, positives = 5-star holds, `seen` built from all train
  interactions including 0-star rows; Wilson/normal 95% CIs quoted. An RMSE-trained SVD
  is not automatically a ranker — the printout quantifies the gap to popularity.
- **Decision recorded** — `config.yaml → data.zero_rating_policy = exclude` (0 = comment
  without stars, per EDA §9) is asserted in §0 and implemented in §1.


In [11]:
def res(regime, model, metric):
    row = results[(results.regime == regime) & (results.model == model)]
    return float(row[metric].iloc[0])

cfg_nf, cfg_reg = CFG["svd"]["n_factors"], CFG["svd"]["reg_all"]
cfg_row = grid[(grid.n_factors == cfg_nf) & (grid.reg_all == cfg_reg)]
cfg_note = (f"config.yaml ({cfg_nf}, {cfg_reg}) scored {float(cfg_row['inner RMSE'].iloc[0]):.4f}"
            if len(cfg_row) else f"config.yaml ({cfg_nf}, {cfg_reg}) was not in the searched grid")

lines = []
lines.append(
    f"Modeled training set: {len(starred):,} starred rows; five-star share "
    f"{FIVE_STAR_SHARE:.1f}% — the skew every number below lives in.")
lines.append(
    f"Grid winner: n_factors={BEST_PARAMS['n_factors']}, reg_all={BEST_PARAMS['reg_all']} "
    f"(inner RMSE {float(BEST['inner RMSE']):.4f}; interior={WINNER_INTERIOR}; {cfg_note}).")
lines.append(
    f"Warm holdout (random): SVD RMSE {res('warm holdout', 'SVD', 'RMSE'):.4f} / MAE "
    f"{res('warm holdout', 'SVD', 'MAE'):.4f}; bias-only "
    f"{res('warm holdout', 'bias only (ALS)', 'RMSE'):.4f} / "
    f"{res('warm holdout', 'bias only (ALS)', 'MAE'):.4f}; best naive = item mean (shrunk) "
    f"{res('warm holdout', 'item mean (shrunk)', 'RMSE'):.4f} / "
    f"{res('warm holdout', 'item mean (shrunk)', 'MAE'):.4f}. SVD-vs-bias gap = "
    f"{res('warm holdout', 'bias only (ALS)', 'RMSE') - res('warm holdout', 'SVD', 'RMSE'):+.4f} "
    f"RMSE (positive = latent factors add real structure).")

t = temporal_results.set_index("model")
svd_delta = float(t.loc["SVD", "temporal RMSE"] - t.loc["SVD", "random RMSE"])
direction = ("harder — the random split overstates warm accuracy for a system claiming to "
             "model evolving preferences") if svd_delta > 0 else \
            "NOT harder on this data — the leakage concern did not materialize"
lines.append(
    f"Temporal vs random warm holdout: SVD RMSE {t.loc['SVD', 'temporal RMSE']:.4f} vs "
    f"{t.loc['SVD', 'random RMSE']:.4f} (MAE {t.loc['SVD', 'temporal MAE']:.4f} vs "
    f"{t.loc['SVD', 'random MAE']:.4f}); bias-only RMSE "
    f"{t.loc['bias only (ALS)', 'temporal RMSE']:.4f} vs "
    f"{t.loc['bias only (ALS)', 'random RMSE']:.4f}. {PCT_LATER:.1f}% of random-holdout "
    f"targets have strictly-later same-user train interactions; the temporal split is "
    f"{direction}.")

for split in ("cold (provided validation)", "cold (provided test)"):
    svd_r, svd_m = res(split, "SVD", "RMSE"), res(split, "SVD", "MAE")
    um_r, um_m = res(split, "user mean", "RMSE"), res(split, "user mean", "MAE")
    b_r, b_m = res(split, "bias only (ALS)", "RMSE"), res(split, "bias only (ALS)", "MAE")
    mae_verdict = (f"raw user mean WINS cold MAE ({um_m:.4f} vs SVD {svd_m:.4f}) — "
                   f"SVD's cold edge is RMSE-only, driven by bias shrinkage"
                   if um_m < svd_m else
                   f"SVD wins cold MAE ({svd_m:.4f} vs user mean {um_m:.4f})")
    lines.append(
        f"{split}: SVD RMSE {svd_r:.4f} / MAE {svd_m:.4f}; bias-only {b_r:.4f} / {b_m:.4f}; "
        f"user mean {um_r:.4f} / {um_m:.4f}. {mae_verdict}. Item identity contributes "
        f"nothing cold — that gap is the Bayesian layer's target (nb 03/04).")

rm_idx = ranking_metrics.set_index("model")
rank_bits = []
for m in rm_idx.index:
    rank_bits.append(
        f"{m}: HR@10 {rm_idx.loc[m, 'hr_at_10']:.3f} "
        f"[{rm_idx.loc[m, 'hr_lo95']:.3f}, {rm_idx.loc[m, 'hr_hi95']:.3f}], NDCG@10 "
        f"{rm_idx.loc[m, 'ndcg_at_10']:.3f} "
        f"[{rm_idx.loc[m, 'ndcg_lo95']:.3f}, {rm_idx.loc[m, 'ndcg_hi95']:.3f}]")
lines.append(
    f"Ranking ({rm_idx['protocol'].iloc[0]}; n={int(rm_idx['n_users'].iloc[0]):,} users; "
    f"95% CIs) — " + "; ".join(rank_bits) + ". Popularity's lead means the hybrid should "
    f"treat exposure as a signal alongside SVD scores rather than assume SVD ranks alone.")

print("TAKEAWAYS (computed this run)")
for i, ln in enumerate(lines, 1):
    print(f"\n{i}. {ln}")


TAKEAWAYS (computed this run)

1. Modeled training set: 681,944 starred rows; five-star share 76.0% — the skew every number below lives in.

2. Grid winner: n_factors=25, reg_all=0.2 (inner RMSE 0.6793; interior=False; config.yaml (50, 0.1) scored 0.6815).

3. Warm holdout (random): SVD RMSE 0.6936 / MAE 0.4393; bias-only 0.6958 / 0.4434; best naive = item mean (shrunk) 0.7175 / 0.4762. SVD-vs-bias gap = +0.0022 RMSE (positive = latent factors add real structure).

4. Temporal vs random warm holdout: SVD RMSE 0.6923 vs 0.6936 (MAE 0.4306 vs 0.4393); bias-only RMSE 0.6945 vs 0.6958. 75.1% of random-holdout targets have strictly-later same-user train interactions; the temporal split is NOT harder on this data — the leakage concern did not materialize.

5. cold (provided validation): SVD RMSE 0.8658 / MAE 0.5793; bias-only 0.8722 / 0.5890; user mean 0.8876 / 0.5493. raw user mean WINS cold MAE (0.5493 vs SVD 0.5793) — SVD's cold edge is RMSE-only, driven by bias shrinkage. Item identity c